# Step 12: Continual Learning & Model Governance
## Controlled Model-Update Process & Champion-Challenger Promotion Gate

### Core Principle:
Do **NOT** blindly retrain whenever performance has a minor hiccup. Retraining should follow a disciplined, audited workflow:
1. Current model (`model_v1`, Champion) is serving predictions.
2. Performance monitor triggers an update alert due to sustained degradation or regular scheduled retrain window.
3. Train candidate model (`model_v2`, Challenger) on an updated sliding window.
4. Evaluate Challenger vs Champion on the exact same holdout evaluation set.
5. **Promotion Gate**: Promote Challenger to Champion **ONLY** if:
   - Overall MAE improves by $\ge 3\%$.
   - High-demand zone MAE does not regress by $> 2\%$.
6. Update the Model Registry with full lineage, training window, validation metrics, and audit decision.


In [ ]:
import sys
from pathlib import Path
import pandas as pd
import numpy as np
import lightgbm as lgb
import joblib

sys.path.append(str(Path.cwd().parent))
from src.data_loader import load_processed_demand, split_chronological
from src.features import build_feature_pipeline, get_feature_columns
from src.continual import ModelRegistry, evaluate_champion_challenger
from src.config import MODELS_DIR

print("Loading current Champion (model_v1) and Model Registry...")
registry = ModelRegistry()
champion_meta = registry.get_champion()
print("Current Champion Metadata:")
print(champion_meta)

champion_model = joblib.load(MODELS_DIR / "model_v1.joblib")
feature_cols = get_feature_columns()


### 1. Continual Learning Trigger & Sliding Window Retraining
To adapt to recent demand patterns, the challenger model is trained on an expanded window including recent data (Jan 8 through Jan 25).
We evaluate both models on the first 3 days of the Test set (Jan 26–28) as the holdout challenge period.


In [ ]:
grid_df = load_processed_demand()
feat_df = build_feature_pipeline(grid_df, drop_burn_in=True)

# Define updated training window for Challenger (Jan 8 to Jan 25)
challenger_train_mask = (feat_df['timestamp'] >= '2025-01-08') & (feat_df['timestamp'] <= '2025-01-25 23:59:59')
challenger_train_df = feat_df[challenger_train_mask]

# Define common holdout evaluation period (Jan 26 to Jan 28)
holdout_mask = (feat_df['timestamp'] >= '2025-01-26') & (feat_df['timestamp'] <= '2025-01-28 23:59:59')
holdout_df = feat_df[holdout_mask]

print(f"Challenger Training Data: {len(challenger_train_df):,} rows ({challenger_train_df['timestamp'].min()} to {challenger_train_df['timestamp'].max()})")
print(f"Common Holdout Eval Data: {len(holdout_df):,} rows ({holdout_df['timestamp'].min()} to {holdout_df['timestamp'].max()})")


### 2. Train Challenger Model (model_v2)


In [ ]:
X_chall_train = challenger_train_df[feature_cols]
y_chall_train = challenger_train_df['demand'].to_numpy()

challenger_params = {
    'objective': 'regression_l1',
    'metric': 'mae',
    'boosting_type': 'gbdt',
    'n_estimators': 300,
    'learning_rate': 0.08,
    'num_leaves': 63,
    'max_depth': 8,
    'subsample': 0.8,
    'colsample_bytree': 0.8,
    'random_state': 42,
    'n_jobs': -1,
    'verbose': -1
}

challenger_model = lgb.LGBMRegressor(**challenger_params)
print("Fitting Challenger Model (model_v2)...")
challenger_model.fit(X_chall_train, y_chall_train)
print("Challenger training complete.")


### 3. Champion vs Challenger Evaluation Gate
Evaluate both models on the common holdout period.
Criteria:
- Overall MAE improvement $\ge 3\%$.
- High-demand zone MAE does not regress by $> 2\%$.

We identify top 20% high-demand zones to verify safety.


In [ ]:
top_zones = (
    challenger_train_df.groupby('PULocationID')['demand'].sum()
    .sort_values(ascending=False)
    .head(52)
    .index.tolist()
)

promoted, eval_summary = evaluate_champion_challenger(
    champion_model=champion_model,
    challenger_model=challenger_model,
    holdout_df=holdout_df,
    feature_cols=feature_cols,
    high_demand_zones=top_zones,
    min_improvement=0.03
)

print("Champion vs Challenger Evaluation Gate Results:")
for k, v in eval_summary.items():
    print(f"  - {k:<28}: {v}")


### 4. Controlled Model Promotion & Registry Update
If the challenger satisfies all criteria, it is promoted to production Champion, and the previous Champion is safely archived.


In [ ]:
model_v2_path = MODELS_DIR / "model_v2.joblib"
joblib.dump(challenger_model, model_v2_path)

if promoted:
    print("PROMOTION CRITERIA MET! Promoting model_v2 to Champion...")
    registry.register_model(
        version="model_v2",
        model_type="LightGBM Regressor (Retrained)",
        training_period=("2025-01-08", "2025-01-25"),
        features=feature_cols,
        val_metrics={"MAE": eval_summary["challenger_mae"]},
        status="champion",
        reason=f"Promoted over model_v1: +{eval_summary['pct_overall_improvement']}% overall MAE improvement",
        artifact_path=str(model_v2_path)
    )
    registry.promote_to_champion("model_v2", reason="Beats model_v1 on holdout validation gate")
else:
    print("Promotion criteria not met. Retaining model_v1 as Champion.")
    registry.register_model(
        version="model_v2",
        model_type="LightGBM Regressor (Retrained)",
        training_period=("2025-01-08", "2025-01-25"),
        features=feature_cols,
        val_metrics={"MAE": eval_summary["challenger_mae"]},
        status="rejected",
        reason="Did not achieve required improvement margin over champion",
        artifact_path=str(model_v2_path)
    )

print("\nUpdated Model Registry Lineage Table:")
print(registry.get_lineage_table().to_string(index=False))


### Summary of Continual Learning
1. **Audited Governance**: Every candidate model is logged with exact training dates, features, and validation metrics.
2. **Safety Gates**: Prevents silent performance regression on high-value business zones.
3. **Reproducibility**: Models are versioned and can be rolled back instantly in `model_registry.json`.
